# Dual-View SARF (gated fusion) Cross-Validation on the Development Pool

As above, but with a learned per-token fusion weight instead of a fixed average.

Cross-validation runs over the **development pool (80% of the corpus)**. The
remaining 20% sit in `splits/test_LOCKED.csv` and are read only by
`test_evaluation_v2`.

Every row of the development pool is predicted exactly once while it was not in
training. Fold assignment is group-aware (identical texts stay together) and
is read from `splits/dev_folds.csv` so that all three architectures see the same
folds.

Hyperparameters, including the number of epochs, come from
`hpo/dual_view_gated_v2/best_params.json`. There is no early stopping: each fold trains
for the tuned number of epochs, and the held-out fold is scored once afterwards.

**Outputs**

- `cv/dual_view_gated_v2/dual_view_gated_v2_fold{1..5}.pth` — the five fold models
- `cv/dual_view_gated_v2/oof_predictions.csv` — out-of-fold predictions and probabilities
- `cv/dual_view_gated_v2/cv_summary.csv` — per-fold macro-F1
- `cv/dual_view_gated_v2/dual_view_gated_v2_misclassified.csv` — every error, sorted by confidence
- `final_models/dual_view_gated_v2_final.pth` — model retrained on the full development pool


In [ ]:
!pip install -q transformers sentencepiece accelerate

In [ ]:
import re
import json
import os
import random
import sys

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
from models import *
import utils_split as u

VARIANT  = "dual_view_gated_v2"
SAVE_DIR = cv_dir(VARIANT)
print("Variant:", VARIANT)
print("Output :", SAVE_DIR)

BERT_MODEL_NAME = "xlm-roberta-large"
MAX_LEN         = 128

In [ ]:
# Hyperparameters are taken from the Optuna search. EPOCHS matters most here:
# without early stopping, the number of epochs has to come from the search.
with open(os.path.join(hpo_dir(VARIANT), "best_params.json"), encoding="utf-8") as f:
    P = json.load(f)

BATCH_SIZE   = P["batch_size"]
ENCODER_LR   = P["encoder_lr"]
HEAD_LR      = P["head_lr"]
DROPOUT      = P["dropout"]
WEIGHT_DECAY = P["weight_decay"]
WARMUP_RATIO = P["warmup_ratio"]
EPOCHS       = P["epochs"]

print("Hyperparameters from best_params.json:")
for k, v in P.items():
    if not k.startswith("_"):
        print(f"  {k:14s} = {v}")
print(f"\nBest value on the HPO slice: {P['_best_value']:.4f} "
      f"({P['_n_trials']} trials)")

In [ ]:
SEED = TRAIN_SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Seed:", SEED)

In [ ]:
# The fold number is already stored in dev_folds.csv, so all three
# architectures train and score on exactly the same partitions.
cv_df = pd.read_csv(DEV_POOL, encoding="utf-8")
cv_df = cv_df.loc[:, ~cv_df.columns.str.contains("^Unnamed")]
cv_df = cv_df.dropna(subset=["surface", "lemma"]).reset_index(drop=True)
folds_df = pd.read_csv(DEV_FOLDS, encoding="utf-8")
cv_df = cv_df.merge(folds_df, on="row_id", how="left", suffixes=("", "_f"))
if "fold_f" in cv_df.columns:
    cv_df = cv_df.drop(columns=["fold_f"])
assert cv_df["fold"].notna().all(), "rows without a fold assignment"
cv_df["fold"] = cv_df["fold"].astype(int)
cv_df = cv_df.reset_index(drop=True)

with open(MANIFEST, encoding="utf-8") as f:
    MAN = json.load(f)
print(f"Development pool: {len(cv_df)} rows (manifest: {MAN['n_dev']})")
assert len(cv_df) == MAN["n_dev"], "development pool differs from the manifest"

print("Label distribution:", cv_df["label"].value_counts().sort_index().to_dict())
print("Fold sizes        :", cv_df["fold"].value_counts().sort_index().tolist())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

## Training and scoring

In [ ]:
# Training and scoring are kept apart on purpose.
#
# train_model() trains for the tuned number of epochs and never sees the
# held-out fold. predict() scores that fold once, afterwards, with the finished
# model. Because no checkpoint is selected by looking at the held-out fold, the
# reported fold score is an unbiased estimate.
#
# The same two functions are reused for the final retraining on the full pool.

def train_model_guarded(train_df, seed, tag=""):
    # XLM-R-large diverges from some initializations. Retry with a bumped seed
    # rather than letting a collapsed run poison the OOF / test predictions.
    for attempt in range(COLLAPSE_RETRIES):
        s = seed + 1000 * attempt
        try:
            return train_model(train_df, s, tag=tag), s
        except CollapseError as e:
            print(f"{tag}[collapse guard] {e} - retry {attempt+1}/{COLLAPSE_RETRIES}")
            torch.cuda.empty_cache()
    raise RuntimeError(f"{tag}training collapsed {COLLAPSE_RETRIES} times in a row")

def build_model():
    # gated=True is essential: the shared class in models.py defaults to average
    # fusion, and the gate is exactly what this variant exists to study. Without
    # it the run would silently train dual_view_v2 a second time.
    return DualViewCNNBiLSTMAttention(
        bert_model_name=BERT_MODEL_NAME, num_classes=3, dropout=DROPOUT,
        gated=True
    )

def make_loader(df, shuffle):
    # LEMMA_COL routes the second branch: "lemma" for the dual-view variants,
    # "surface" for the capacity-matched control, which sees the same view twice.
    return DataLoader(DualViewSurfaceLemmaDataset(df, tokenizer,
                                                  lemma_col=LEMMA_COL[VARIANT]),
                      batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dual_surface_lemma_collator)

def train_model(train_df, seed, tag=""):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_loader = make_loader(train_df, shuffle=True)

    model = build_model()
    model.to(device)

    encoder_params = list(model.encoder.parameters())
    head_params    = [p for n, p in model.named_parameters() if "encoder" not in n]
    optimizer = optim.AdamW([
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params,    "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"{tag}Epoch {epoch+1}/{EPOCHS}", leave=False):
            surface = {k: v.to(device) for k, v in batch["surface"].items()}
            lemma   = {k: v.to(device) for k, v in batch["lemma"].items()}
            labels  = batch["labels"].to(device)

            optimizer.zero_grad()
            logits, _ = model(surface, lemma)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_train_loss += loss.item()
        mean_loss = total_train_loss / len(train_loader)
        print(f"{tag}Epoch {epoch+1}/{EPOCHS} | train loss: {mean_loss:.4f}")
        # Collapse guard: bail out after the first epoch if the run diverged.
        if epoch == 0 and mean_loss > COLLAPSE_LOSS:
            raise CollapseError(
                f"{tag}epoch-1 loss {mean_loss:.4f} > ln(3)={COLLAPSE_LOSS:.4f}")

    del optimizer, scheduler
    torch.cuda.empty_cache()
    return model

def predict(model, df):
    loader = make_loader(df, shuffle=False)
    model.eval()
    preds, labels_out, probs, sw_out = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            surface = {k: v.to(device) for k, v in batch["surface"].items()}
            lemma   = {k: v.to(device) for k, v in batch["lemma"].items()}
            labels  = batch["labels"].to(device)

            logits, surface_weight = model(surface, lemma)
            p = F.softmax(logits, dim=1)
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_out.extend(labels.cpu().numpy())
            probs.extend(p.cpu().numpy().tolist())

            # mean gate weight per sample over non-padding tokens
            sw    = surface_weight.squeeze(-1)
            smask = surface["attention_mask"].float()
            per_sample = (sw * smask).sum(dim=1) / smask.sum(dim=1).clamp(min=1e-9)
            sw_out.extend(per_sample.cpu().numpy().tolist())
    return (np.array(preds), np.array(labels_out),
            np.array(probs), np.array(sw_out))

## Cross-validation

In [ ]:
fold_f1s = []
fold_seeds_used = []
fold_surface_means = []
epochs_used = []

oof_preds  = np.zeros(len(cv_df), dtype=int)
oof_labels = np.zeros(len(cv_df), dtype=int)
oof_probs  = np.zeros((len(cv_df), 3), dtype=float)
oof_surface_w = np.zeros(len(cv_df), dtype=float)

for fold in range(N_FOLDS):
    print(f"\n{'='*60}\nFOLD {fold+1}/{N_FOLDS}\n{'='*60}")
    tr_idx = np.where(cv_df["fold"].values != fold)[0]
    va_idx = np.where(cv_df["fold"].values == fold)[0]

    fold_train_df = cv_df.iloc[tr_idx].reset_index(drop=True)
    fold_val_df   = cv_df.iloc[va_idx].reset_index(drop=True)
    print(f"Train: {len(fold_train_df)} | Held-out fold: {len(fold_val_df)}")

    fold_seed = SEED + fold
    model, used_seed = train_model_guarded(fold_train_df, fold_seed, tag=f"Fold {fold+1} | ")
    fold_seeds_used.append(used_seed)

    # The held-out fold is touched only here, exactly once.
    preds, labels, probs, sweights = predict(model, fold_val_df)

    fold_f1 = f1_score(labels, preds, average="macro")
    fold_f1s.append(fold_f1)
    epochs_used.append(EPOCHS)

    oof_preds[va_idx]  = preds
    oof_labels[va_idx] = labels
    oof_probs[va_idx]  = probs
    oof_surface_w[va_idx] = sweights
    fold_surface_means.append(float(np.mean(sweights)))

    torch.save(model.state_dict(),
               os.path.join(SAVE_DIR, f"{VARIANT}_fold{fold+1}.pth"))
    del model
    torch.cuda.empty_cache()

    print(f"\nFold {fold+1} macro-F1: {fold_f1:.4f} | surface weight: {np.mean(sweights)*100:.2f}%")

## Results

In [ ]:
fold_f1s = np.array(fold_f1s)
print("===== Macro-F1 per fold =====")
for i, f in enumerate(fold_f1s):
    print(f"  Fold {i+1}: {f:.4f}")
print("-" * 45)
print(f"  Mean: {fold_f1s.mean():.4f}")
print(f"  Std : {fold_f1s.std():.4f}")
print(f"  Min : {fold_f1s.min():.4f}")
print(f"  Max : {fold_f1s.max():.4f}")

target_names = ["Neutral (0)", "Negative (1)", "Positive (2)"]
print("\n===== Pooled out-of-fold report (development pool) =====")
print(classification_report(oof_labels, oof_preds, target_names=target_names,
                            labels=[0, 1, 2]))

oof_macro_f1 = f1_score(oof_labels, oof_preds, average="macro")
print(f"Pooled OOF macro-F1 (development pool, n={len(cv_df)}): {oof_macro_f1:.4f}")

pd.DataFrame({
    "fold": list(range(1, N_FOLDS + 1)),
    "macro_f1": fold_f1s,
    "epochs": epochs_used,
    "train_seed": fold_seeds_used,
}).to_csv(os.path.join(SAVE_DIR, "cv_summary.csv"), index=False)

# Out-of-fold predictions drive every analysis in the error section and the
# comparison between architectures, so they are written out in full.
pd.DataFrame({
    "row_id": cv_df["row_id"].values,
    "fold":   cv_df["fold"].values,
    "label":  oof_labels,
    "pred":   oof_preds,
    "p0": oof_probs[:, 0], "p1": oof_probs[:, 1], "p2": oof_probs[:, 2],
    "surface_weight": oof_surface_w,
}).to_csv(os.path.join(SAVE_DIR, "oof_predictions.csv"), index=False, encoding="utf-8")
print("saved:", os.path.join(SAVE_DIR, "oof_predictions.csv"))

In [ ]:
cm   = confusion_matrix(oof_labels, oof_preds, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Neutral", "Negative", "Positive"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Out-of-fold confusion matrix - {N_FOLDS}-fold CV, development pool")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "cv_oof_confusion_matrix.png"), dpi=150)
plt.show()

# Error analysis

Based on the pooled out-of-fold predictions over the development pool. Every row
was predicted exactly once while it was not in training, and no checkpoint was
selected by looking at those rows.

The test set does not appear anywhere in this section.

In [ ]:
label_names = ["Neutral", "Negative", "Positive"]

err_df = cv_df.copy().reset_index(drop=True)
err_df["true"]       = oof_labels
err_df["pred"]       = oof_preds
err_df["confidence"] = oof_probs.max(axis=1)
err_df["p_true"]     = oof_probs[np.arange(len(err_df)), oof_labels]
err_df["correct"]    = err_df["true"] == err_df["pred"]
err_df["true_name"]  = [label_names[i] for i in err_df["true"]]
err_df["pred_name"]  = [label_names[i] for i in err_df["pred"]]
err_df["text_len"]   = err_df["surface"].astype(str).str.split().apply(len)

n_total = len(err_df)
n_wrong = int((~err_df["correct"]).sum())
print(f"Rows (development pool): {n_total}")
print(f"Incorrect              : {n_wrong}  ({100*n_wrong/n_total:.2f}%)")
print(f"Accuracy               : {100*err_df['correct'].mean():.2f}%")

In [ ]:
print("===== Error rate per class =====")
for c, name in enumerate(label_names):
    mask  = err_df["true"] == c
    n_c   = int(mask.sum())
    err_c = int((mask & ~err_df["correct"]).sum())
    print(f"  {name:9s}: {err_c:4d}/{n_c:4d} wrong  ({100*err_c/max(n_c,1):.2f}%)")

print("\n===== Most frequent confusions (true -> predicted) =====")
conf_pairs = (err_df[~err_df["correct"]]
              .groupby(["true_name", "pred_name"]).size()
              .sort_values(ascending=False))
for (t, p), n in conf_pairs.items():
    print(f"  {t:9s} -> {p:9s}: {n}")

In [ ]:
TOP_K = 20
worst = (err_df[~err_df["correct"]]
         .sort_values("confidence", ascending=False)
         .head(TOP_K))

pd.set_option("display.max_colwidth", 120)
print(f"===== Top {TOP_K} most confident errors =====\n")
for _, row in worst.iterrows():
    print(f"[conf {row['confidence']:.2f}]  true={row['true_name']:9s}  "
          f"pred={row['pred_name']:9s}")
    print(f"   {row['surface']}")
    print()

In [ ]:
bins       = [0, 5, 10, 20, 40, np.inf]
bucket_lbl = ["1-5", "6-10", "11-20", "21-40", "40+"]
err_df["len_bucket"] = pd.cut(err_df["text_len"], bins=bins, labels=bucket_lbl)

by_len = err_df.groupby("len_bucket", observed=True).agg(
    n=("correct", "size"),
    err_rate_pct=("correct", lambda s: 100 * (1 - s.mean())),
    mean_conf=("confidence", "mean"),
)
print("===== Error rate by text length (words) =====")
print(by_len.round(2))

In [ ]:
# Denominators are the category totals inside the development pool.
if "category" in err_df.columns:
    by_cat = (err_df.groupby("category", observed=True)
              .agg(n=("correct", "size"),
                   err_rate_pct=("correct", lambda s: 100 * (1 - s.mean())),
                   mean_conf=("confidence", "mean"))
              .sort_values("err_rate_pct", ascending=False))
    print("===== Error rate per category (development pool) =====")
    print(by_cat.round(2))

    # Small categories cannot carry a reliable rate; technology drops to
    # roughly 135 rows in the development pool.
    small = by_cat[by_cat["n"] < 200]
    if len(small):
        print("\nToo small for a reliable rate:",
              ", ".join(f"{i} (n={int(r.n)})" for i, r in small.iterrows()))

    by_cat.to_csv(os.path.join(SAVE_DIR, "error_rate_by_category.csv"),
                  encoding="utf-8")

    print("\n===== Confusions per category (true -> pred) =====")
    for cat, sub in err_df[~err_df["correct"]].groupby("category", observed=True):
        pairs = sub.groupby(["true_name", "pred_name"]).size().sort_values(ascending=False)
        top = "; ".join(f"{t}->{p}:{n}" for (t, p), n in pairs.head(3).items())
        print(f"  {cat:11s} ({len(sub)} errors): {top}")
else:
    print("No 'category' column in the development pool - breakdown skipped.")

In [ ]:
# row_id is exported so that errors can be paired across architectures.
mis_cols = ["row_id", "surface"]
if "category" in err_df.columns:
    mis_cols.append("category")
mis_cols += ["true_name", "pred_name", "confidence", "p_true", "text_len"]

mis_df = (err_df.loc[~err_df["correct"], mis_cols]
          .sort_values("confidence", ascending=False)
          .reset_index(drop=True))

mis_path = os.path.join(SAVE_DIR, f"{VARIANT}_misclassified.csv")
mis_df.to_csv(mis_path, index=False, encoding="utf-8")
print(f"{len(mis_df)} misclassified rows saved to {mis_path}")
mis_df.head(15)

## Fusion gate

In [ ]:
# Out-of-fold behaviour of the learned fusion gate.
print("===== Mean surface weight per fold =====")
for i, m in enumerate(fold_surface_means):
    print(f"  Fold {i+1}: surface {m*100:5.2f}%  |  lemma {(1-m)*100:5.2f}%")

oof_surface_mean = float(oof_surface_w.mean())
print("\n===== Overall weighting (out-of-fold, development pool) =====")
print(f"  Surface: {oof_surface_mean*100:.2f} %")
print(f"  Lemma  : {(1-oof_surface_mean)*100:.2f} %")
print(f"  Std across samples: {oof_surface_w.std()*100:.2f} pp")

pd.DataFrame({
    "row_id": cv_df["row_id"].values,
    "surface": cv_df["surface"],
    "label": oof_labels,
    "oof_pred": oof_preds,
    "surface_weight": oof_surface_w,
}).to_csv(os.path.join(SAVE_DIR, "oof_surface_weights.csv"),
          index=False, encoding="utf-8")
print("\nsaved:", os.path.join(SAVE_DIR, "oof_surface_weights.csv"))

# Final retraining (optional)

**Not required for the reported test numbers.** Those come from the five fold
models scored on the locked test set inside `test_evaluation_v2`, which yields a
genuine dispersion estimate. A single full-pool model has no dispersion and
cannot enter the seed-paired test.

Set `RUN_FILE` note: `RUN_FINAL = True` only if a single full-data checkpoint per
architecture is wanted (e.g. for comparability with prior work). It costs one
extra run per variant and is reported as an additional row, not the headline.

In [ ]:
RUN_FINAL = False

if RUN_FINAL:
    print(f"Training on all {len(cv_df)} development rows for {EPOCHS} epochs ...")
    fmodel, _ = train_model_guarded(cv_df, SEED, tag="Final | ")

    out_path = final_model(VARIANT)
    torch.save(fmodel.state_dict(), out_path)
    print("\nsaved:", out_path)

    meta = {
        "variant": VARIANT,
        "trained_on": "dev_pool",
        "n_train": int(len(cv_df)),
        "epochs": int(EPOCHS),
        "seed": int(SEED),
        "hyperparams": {k: v for k, v in P.items() if not k.startswith("_")},
        "cv_mean_macro_f1": float(fold_f1s.mean()),
        "cv_std_macro_f1": float(fold_f1s.std()),
        "oof_macro_f1": float(oof_macro_f1),
        "dev_pool_sha256": MAN["dev_pool_sha256"],
    }
    with open(os.path.join(SAVE_DIR, "final_model_meta.json"), "w",
              encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(json.dumps(meta, indent=2))

    del fmodel
    torch.cuda.empty_cache()
else:
    print("RUN_FINAL = False - skipped.")

In [ ]:
assert_test_untouched(globals())

print("\nSummary")
print(f"  Variant        : {VARIANT}")
print(f"  CV macro-F1    : {fold_f1s.mean():.4f} +/- {fold_f1s.std():.4f}")
print(f"  Pooled OOF F1  : {oof_macro_f1:.4f}  (n={len(cv_df)})")
print(f"  Fold models    : {SAVE_DIR}")
print(f"  Final model    : {final_model(VARIANT)}")